# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [9]:
%reset -f

import gc
gc.collect()

0

In [10]:
import os
import boto3
from dotenv import load_dotenv
import sys
import scipy.sparse
from IPython.display import display
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

load_dotenv()

True

In [11]:
import utils
import importlib

importlib.reload(utils)

<module 'utils' from '/home/mle-user/sprint5/final_project/utils.py'>

In [12]:
import boto3, os
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket_name = os.getenv("S3_BUCKET_NAME")

In [13]:
# ============================================================
# ПОСТРОЕНИЕ ПРИЗНАКОВ ДЛЯ РАНЖИРУЮЩЕЙ МОДЕЛИ
# ============================================================
import gc
import numpy as np
import pandas as pd



events = pd.read_parquet(
    "parquet_cleaned/events.parquet"
)
# ------------------------------------------------------------
# Проверяем размеры загруженных таблиц
# ------------------------------------------------------------

print("events:", events.shape)


events: (2755641, 5)


In [14]:
# ============================================================
# ПОДГОТОВКА EVENTS
# ============================================================



# Приводим названия к единому виду
events = events.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
    }
)


# Преобразуем timestamp
events["event_time"] = pd.to_datetime(
    events["timestamp"]
)


print(events.columns.tolist())

print(
    "Период данных:",
    events["event_time"].min(),
    "->",
    events["event_time"].max()
)

['timestamp', 'user_id', 'event', 'item_id', 'transactionid', 'event_time']
Период данных: 2015-05-03 03:00:04.384000 -> 2015-09-18 02:59:47.788000


In [15]:

data_end = (
    events["event_time"].max()
    + pd.Timedelta(milliseconds=1)
)

eval_start = (
    data_end
    - pd.Timedelta(days=7)
)

ranker_start = (
    eval_start
    - pd.Timedelta(days=7)
)

# История, которую разрешено использовать
# при построении признаков и кандидатов
events_history = events[
    events["event_time"] < ranker_start
].copy()


print(
    "History:",
    events_history["event_time"].min(),
    "->",
    events_history["event_time"].max()
)

print("Ranker start:", ranker_start)
print("Eval start:", eval_start)

History: 2015-05-03 03:00:04.384000 -> 2015-09-04 02:59:46.763000
Ranker start: 2015-09-04 02:59:47.789000
Eval start: 2015-09-11 02:59:47.789000


In [16]:
import pyarrow as pa
import pyarrow.parquet as pq

input_path = (
    "recommendations/personal_als/"
    "personal_als.parquet"
)

output_path = (
    "recommendations/personal_als/"
    "als_candidates.parquet"
)

parquet_file = pq.ParquetFile(input_path)

writer = None

try:

    for row_group_idx in range(
        parquet_file.num_row_groups
    ):

        batch = (
            parquet_file
            .read_row_group(
                row_group_idx,
                columns=[
                    "visitorid",
                    "itemid"
                ]
            )
            .to_pandas()
            )

        batch = batch.rename(
            columns={
                "visitorid": "user_id",
                "itemid": "item_id"
            }
        )

        batch = batch.drop_duplicates(
            subset=[
                "user_id",
                "item_id"
            ]
        )

        table = pa.Table.from_pandas(
            batch,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        del batch
        del table

        print(
            f"Обработано "
            f"{row_group_idx + 1} / "
            f"{parquet_file.num_row_groups}"
        )

finally:

    if writer is not None:
        writer.close()

Обработано 1 / 128
Обработано 2 / 128
Обработано 3 / 128
Обработано 4 / 128
Обработано 5 / 128
Обработано 6 / 128
Обработано 7 / 128
Обработано 8 / 128
Обработано 9 / 128
Обработано 10 / 128
Обработано 11 / 128
Обработано 12 / 128
Обработано 13 / 128
Обработано 14 / 128
Обработано 15 / 128
Обработано 16 / 128
Обработано 17 / 128
Обработано 18 / 128
Обработано 19 / 128
Обработано 20 / 128
Обработано 21 / 128
Обработано 22 / 128
Обработано 23 / 128
Обработано 24 / 128
Обработано 25 / 128
Обработано 26 / 128
Обработано 27 / 128
Обработано 28 / 128
Обработано 29 / 128
Обработано 30 / 128
Обработано 31 / 128
Обработано 32 / 128
Обработано 33 / 128
Обработано 34 / 128
Обработано 35 / 128
Обработано 36 / 128
Обработано 37 / 128
Обработано 38 / 128
Обработано 39 / 128
Обработано 40 / 128
Обработано 41 / 128
Обработано 42 / 128
Обработано 43 / 128
Обработано 44 / 128
Обработано 45 / 128
Обработано 46 / 128
Обработано 47 / 128
Обработано 48 / 128
Обработано 49 / 128
Обработано 50 / 128
Обработан

In [17]:


duckdb.sql("""
COPY (
    SELECT
        user_id,
        item_id AS source_item_id
    FROM (
        SELECT
            visitorid AS user_id,
            itemid AS item_id,
            timestamp,
            ROW_NUMBER() OVER (
                PARTITION BY visitorid, itemid
                ORDER BY timestamp DESC
            ) AS item_rn
        FROM read_parquet('parquet_cleaned/events.parquet')
        WHERE
            event IN ('view', 'addtocart', 'transaction')
            AND timestamp < ?
    )
    WHERE item_rn = 1
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY timestamp DESC
    ) <= 10
)
TO 'user_history.parquet'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""", params=[ranker_start])

In [18]:
duckdb.sql("""
COPY (
    SELECT
        h.user_id,
        s.similar_itemid AS item_id,
        MAX(s.score) AS similarity_score,
        MIN(s.rank) AS similarity_rank,
        COUNT(DISTINCT h.source_item_id) AS similarity_sources
    FROM read_parquet('user_history.parquet') h
    JOIN read_parquet(
        'recommendations/similar/similar.parquet'
    ) s
        ON h.source_item_id = s.itemid
    WHERE s.rank <= 50
    GROUP BY
        h.user_id,
        s.similar_itemid
)
TO 'similar_features.parquet'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:

duckdb.sql("""
COPY (
    SELECT DISTINCT
        user_id,
        item_id
    FROM (
        SELECT
            visitorid AS user_id,
            itemid AS item_id
        FROM read_parquet(
            'recommendations/personal_als/personal_als.parquet'
        )

        UNION ALL

        SELECT
            user_id,
            item_id
        FROM read_parquet(
            'similar_features.parquet'
        )

        UNION ALL

        SELECT
            u.user_id,
            p.item_id
        FROM (
            SELECT DISTINCT visitorid AS user_id
            FROM read_parquet(
                'parquet_cleaned/events.parquet'
            )
            WHERE timestamp < ?
        ) u
        CROSS JOIN (
            SELECT
                itemid AS item_id
            FROM read_parquet(
                'recommendations/top_popular/top_popular.parquet'
            )
            ORDER BY rank
            LIMIT 10
        ) p
    )
)
TO 'candidates_unique.parquet'
(FORMAT PARQUET, COMPRESSION SNAPPY)
""", params=[ranker_start])



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:

OUTPUT_FILE = "candidates_with_popular.parquet"

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

duckdb.sql("""
COPY (
    SELECT
        CAST(c.user_id AS INTEGER) AS user_id,
        CAST(c.item_id AS INTEGER) AS item_id,

        CAST(p.score AS FLOAT) AS popular_score,
        CAST(p.rank AS SMALLINT) AS popular_rank

    FROM read_parquet(
        'candidates_unique.parquet'
    ) c

    LEFT JOIN read_parquet(
        'recommendations/top_popular/top_popular.parquet'
    ) p
        ON c.item_id = p.itemid
)
TO 'candidates_with_popular.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")

print(
    "Готово:",
    OUTPUT_FILE
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Готово: candidates_with_popular.parquet


In [23]:

OUTPUT_FILE = "candidates_with_similar.parquet"
OLD_FILE = "candidates_with_popular.parquet"
MIN_VALID_SIZE = 1024


# Удаляем недописанный файл прошлой попытки
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)


duckdb.sql("""
COPY (
    SELECT
        CAST(c.user_id AS INTEGER) AS user_id,
        CAST(c.item_id AS INTEGER) AS item_id,

        c.popular_score,
        c.popular_rank,

        CAST(s.similarity_score AS FLOAT) AS similarity_score,
        CAST(s.similarity_rank AS SMALLINT) AS similarity_rank,
        CAST(s.similarity_sources AS SMALLINT) AS similarity_sources

    FROM read_parquet(
        'candidates_with_popular.parquet'
    ) c

    LEFT JOIN read_parquet(
        'similar_features.parquet'
    ) s
        ON c.user_id = s.user_id
       AND c.item_id = s.item_id
)
TO 'candidates_with_similar.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")


print(
    "Готово:",
    OUTPUT_FILE
)


# Удаляем старый файл только если новый успешно создан
if (
    os.path.exists(OUTPUT_FILE)
    and os.path.getsize(OUTPUT_FILE) >= MIN_VALID_SIZE
):
    if (
        OLD_FILE != OUTPUT_FILE
        and os.path.exists(OLD_FILE)
    ):
        os.remove(OLD_FILE)

        print(
            "Удалён старый:",
            OLD_FILE
        )
else:
    print(
        "Ошибка записи — старый файл не удаляю"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Готово: candidates_with_similar.parquet
Удалён старый: candidates_with_popular.parquet


In [24]:
OUTPUT_FILE = "candidates_with_als.parquet"
OLD_FILE = "candidates_with_similar.parquet"
MIN_VALID_SIZE = 1024


# Удаляем недописанный файл прошлой попытки
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)


duckdb.sql("""
COPY (
    SELECT
        c.*,

        CAST(a.score AS FLOAT) AS als_score,
        CAST(a.rank AS SMALLINT) AS als_rank

    FROM read_parquet(
        'candidates_with_similar.parquet'
    ) c

    LEFT JOIN read_parquet(
        'recommendations/personal_als/personal_als.parquet'
    ) a
        ON c.user_id = a.visitorid
       AND c.item_id = a.itemid
)
TO 'candidates_with_als.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")


print(
    "Готово:",
    OUTPUT_FILE
)


# Удаляем старый файл только после успеха
if (
    os.path.exists(OUTPUT_FILE)
    and os.path.getsize(OUTPUT_FILE) >= MIN_VALID_SIZE
):
    if (
        OLD_FILE != OUTPUT_FILE
        and os.path.exists(OLD_FILE)
    ):
        os.remove(OLD_FILE)

        print(
            "Удалён старый:",
            OLD_FILE
        )
else:
    print(
        "Ошибка записи — старый файл не удаляю"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Готово: candidates_with_als.parquet
Удалён старый: candidates_with_similar.parquet


In [25]:

OUTPUT_FILE = "candidates_result.parquet"

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

duckdb.sql("""
COPY (
    SELECT
        user_id,
        item_id,

        CAST(als_score AS FLOAT) AS als_score,
        als_rank,

        CAST(popular_score AS FLOAT) AS popular_score,
        popular_rank,

        CAST(similarity_score AS FLOAT) AS similarity_score,
        similarity_rank,
        CAST(similarity_sources AS SMALLINT) AS similarity_sources,

        CAST(
            CASE
                WHEN als_score IS NOT NULL THEN 1
                ELSE 0
            END
            AS TINYINT
        ) AS from_als,

        CAST(
            CASE
                WHEN popular_score IS NOT NULL THEN 1
                ELSE 0
            END
            AS TINYINT
        ) AS from_popular,

        CAST(
            CASE
                WHEN similarity_score IS NOT NULL THEN 1
                ELSE 0
            END
            AS TINYINT
        ) AS from_similar,

        CAST(
            LN(1 + popular_score)
            AS FLOAT
        ) AS popular_score_log

    FROM read_parquet(
        'candidates_with_als.parquet'
    )
)
TO 'candidates_result.parquet'
(
    FORMAT PARQUET,
    COMPRESSION SNAPPY,
    ROW_GROUP_SIZE 250000
)
""")

print("Готово:", OUTPUT_FILE)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Готово: candidates_result.parquet


In [34]:
import pandas as pd

candidates = pd.read_parquet("candidates_result.parquet")

In [27]:
display(candidates.head(20))

,user_id,item_id,als_score,als_rank,popular_score,popular_rank,similarity_score,similarity_rank,similarity_sources,from_als,from_popular,from_similar,popular_score_log
0,1228945,450082,2.109841e-03,44.0,33.0,51.0,NaN,NaN,NaN,1,1,0,3.526361
1,1228955,65273,2.096621e-02,84.0,39.0,34.0,NaN,NaN,NaN,1,1,0,3.688879
2,1228974,384302,2.321338e-02,47.0,46.0,26.0,NaN,NaN,NaN,1,1,0,3.850147
3,1228983,290999,5.643386e-03,44.0,33.0,54.0,NaN,NaN,NaN,1,1,0,3.526361
4,1228983,48030,5.303166e-03,54.0,86.0,6.0,NaN,NaN,NaN,1,1,0,4.465908
5,1229007,138427,8.916453e-13,4.0,31.0,58.0,NaN,NaN,NaN,1,1,0,3.465736
6,1229014,161623,3.934991e-06,54.0,27.0,80.0,NaN,NaN,NaN,1,1,0,3.332205
7,1229133,29196,1.623270e-02,28.0,107.0,5.0,NaN,NaN,NaN,1,1,0,4.682131
8,1229158,92466,1.836313e-02,53.0,32.0,56.0,NaN,NaN,NaN,1,1,0,3.496508
9,1229173,22436,9.412334e-03,54.0,35.0,46.0,NaN,NaN,NaN,1,1,0,3.583519


In [35]:
display(
    candidates[
        candidates["from_similar"] == 1
    ].head(20)
)

,user_id,item_id,als_score,als_rank,popular_score,popular_rank,similarity_score,similarity_rank,similarity_sources,from_als,from_popular,from_similar,popular_score_log
29895,175823,298560,0.318798,6.0,NaN,NaN,0.875339,39.0,1.0,1,0,1,NaN
29896,175831,454766,0.000123,14.0,NaN,NaN,0.880568,45.0,1.0,1,0,1,NaN
29897,175857,174185,0.361199,16.0,NaN,NaN,0.819151,22.0,1.0,1,0,1,NaN
29898,175987,97890,0.000069,4.0,NaN,NaN,0.981610,5.0,2.0,1,0,1,NaN
29899,176228,403792,0.249196,47.0,NaN,NaN,0.998305,9.0,4.0,1,0,1,NaN
29900,176278,351481,0.102584,2.0,NaN,NaN,0.952254,4.0,1.0,1,0,1,NaN
29901,176359,294307,0.006235,56.0,NaN,NaN,0.925520,32.0,1.0,1,0,1,NaN
29902,176503,14758,0.000072,61.0,NaN,NaN,0.879476,9.0,1.0,1,0,1,NaN
29903,176578,121086,0.007232,90.0,NaN,NaN,0.763776,18.0,1.0,1,0,1,NaN
29904,176809,160878,0.020759,86.0,NaN,NaN,0.977365,10.0,1.0,1,0,1,NaN


In [28]:
print(
    "Кандидатов:",
    len(candidates)
)

print(
    "Попали в top popular:",
    (candidates["popular_score"] > 0).sum()
)

print(
    "Доля популярных:",
    (candidates["popular_score"] > 0).mean()
)

Кандидатов: 217593361
Попали в top popular: 26725165
Доля популярных: 0.12282160116089204


In [30]:
non_zero = (candidates != 0).sum()
total = len(candidates)

result = pd.DataFrame({
    "non_zero": non_zero,
    "total": total,
    "percent": (non_zero / total * 100).round(2),
})

print(result)

                     non_zero      total  percent
user_id             217593361  217593361   100.00
item_id             217593361  217593361   100.00
als_score           217593361  217593361   100.00
als_rank            217593361  217593361   100.00
popular_score       217593361  217593361   100.00
popular_rank        217593361  217593361   100.00
similarity_score    217593361  217593361   100.00
similarity_rank     217593361  217593361   100.00
similarity_sources  217593361  217593361   100.00
from_als            128032900  217593361    58.84
from_popular         26725165  217593361    12.28
from_similar         82932659  217593361    38.11
popular_score_log   217593361  217593361   100.00


In [31]:
stats = pd.DataFrame({
    "notna": candidates.notna().sum(),
    "isna": candidates.isna().sum(),
    "zero": (candidates == 0).sum(),
    "non_zero_real": ((candidates != 0) & candidates.notna()).sum(),
})

stats["non_zero_%"] = (stats["non_zero_real"] / len(candidates) * 100).round(2)
print(stats)

                        notna       isna       zero  non_zero_real  non_zero_%
user_id             217593361          0          0      217593361      100.00
item_id             217593361          0          0      217593361      100.00
als_score           128032900   89560461          0      128032900       58.84
als_rank            128032900   89560461          0      128032900       58.84
popular_score        26725165  190868196          0       26725165       12.28
popular_rank         26725165  190868196          0       26725165       12.28
similarity_score     82932659  134660702          0       82932659       38.11
similarity_rank      82932659  134660702          0       82932659       38.11
similarity_sources   82932659  134660702          0       82932659       38.11
from_als            217593361          0   89560461      128032900       58.84
from_popular        217593361          0  190868196       26725165       12.28
from_similar        217593361          0  134660702 

In [33]:
%reset -f

import gc
gc.collect()

0